# FAISS 向量库简介

FAISS（Facebook AI Similarity Search）是一个用于**高效搜索相似向量**的开源库。它并不是传统意义上存储姓名、价格等字段的数据库，而是把文本、图片、音频等数据经过模型转换成向量（Embedding），再根据向量之间的距离寻找最相似的内容。

## 1. FAISS 能解决什么问题？

例如，在知识库问答中，可以先把每段文档转换成向量并存入 FAISS；用户提问时，也把问题转换成向量，然后检索距离最近的若干段文档，最后把这些文档交给大语言模型生成回答。这就是 RAG（检索增强生成）中常见的检索环节。

**基本流程：** 原始数据 → Embedding 模型 → 向量 → 建立 FAISS 索引 → 输入查询向量 → 返回 Top-K 相似结果。

## 2. 常见索引类型

| 索引 | 特点 | 适用场景 |
|---|---|---|
| `IndexFlatL2` | 精确搜索，使用欧氏距离，不需要训练 | 数据量较小、希望结果准确 |
| `IndexFlatIP` | 精确搜索，使用向量内积；向量归一化后可用于余弦相似度 | 文本语义检索 |
| `IndexIVFFlat` | 先聚类再搜索，速度更快，但需要训练索引 | 中大型数据集 |
| `IndexHNSWFlat` | 基于图的近似搜索，速度快、召回率较高 | 大规模在线检索 |

## 3. 最小示例

```python
import faiss
import numpy as np

# 假设有 1000 条数据，每条数据用 128 维向量表示
d = 128
vectors = np.random.random((1000, d)).astype("float32")
queries = np.random.random((2, d)).astype("float32")

# 创建使用欧氏距离的精确索引，并加入向量
index = faiss.IndexFlatL2(d)
index.add(vectors)

# 为每个查询向量查找最相似的 5 个向量
distances, indices = index.search(queries, k=5)
print(indices)    # 相似向量在原数组中的位置
print(distances)  # 对应的距离，L2 距离越小越相似
```

> 注意：FAISS 主要管理向量和向量编号，通常需要另外使用列表、字典或数据库保存编号对应的原文及其他元数据。选择余弦相似度时，应先对向量进行 L2 归一化，再使用 `IndexFlatIP`。

In [ ]:
import faiss
import numpy as np

dimension = 128

# 使用业务ID管理向量
base_index = faiss.IndexFlatIP(dimension)
index = faiss.IndexIDMap2(base_index)

# 增
vector = np.random.rand(1, dimension).astype("float32")
faiss.normalize_L2(vector)

index.add_with_ids(
    vector,
    np.array([1001], dtype="int64")
)

# 查：返回最相似向量的ID和相似度
query = np.random.rand(1, dimension).astype("float32")
faiss.normalize_L2(query)

scores, ids = index.search(query, k=3)

# 删
index.remove_ids(
    np.array([1001], dtype="int64")
)

# 改：删除旧向量，再用相同ID添加新向量
new_vector = np.random.rand(1, dimension).astype("float32")
faiss.normalize_L2(new_vector)

index.add_with_ids(
    new_vector,
    np.array([1001], dtype="int64")
)